# microGPT, on Wikipedia

Recreate Karpathy's microGPT, but trained on real Wikipedia text (`wikimedia/wikipedia`, Simple English) with our own byte-pair-encoding tokenizer and a GPT-2-style Transformer built from raw PyTorch tensor ops (`torch.autograd` for backward).

See `wiki/llm-from-scratch/microgpt.md` and `wiki/llm-from-scratch/nn-zero-to-hero.md`.

## [A] Setup & dependencies

Install deps with `uv` (`torch`, `datasets`) and do **all** of the notebook's imports here, then pick a device (CUDA / MPS / CPU) so the rest of the notebook stays device-agnostic.

In [ ]:
import json
import math
import random
import time
from collections import defaultdict
from pathlib import Path

import torch
from datasets import load_dataset
from huggingface_hub import snapshot_download

# Device-agnostic: prefer CUDA, then Apple MPS, else CPU.
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

# One seed for reproducible runs across data sampling, init, and batching.
torch.manual_seed(1337)

print(f"torch {torch.__version__} | device: {device}")

## [B] Config

All hyperparameters in one place: data subset size, BPE vocab size, `block_size`, `n_layer` / `n_head` / `n_embd`, learning rate, and step count.

In [ ]:
# --- Data ---
n_articles = 2000        # how many articles to sample into the corpus
data_seed = 1337         # seed for the uniform random sample (reproducible corpus)

# --- Tokenizer (BPE) ---
vocab_size = 1024        # final vocab: 256 raw bytes + (vocab_size - 256) learned merges

# --- Model ---
block_size = 128         # context length (tokens the model attends over)
n_layer = 4              # number of Transformer blocks
n_head = 4               # attention heads per block
n_embd = 128             # embedding / residual-stream width (must be divisible by n_head)
assert n_embd % n_head == 0, "n_embd must be divisible by n_head"

# --- Training ---
batch_size = 32          # sequences per step
learning_rate = 3e-4     # peak Adam learning rate
max_steps = 2000         # total optimizer steps
eval_interval = 200      # steps between val-loss checks
eval_iters = 50          # batches averaged per val-loss estimate

print(f"model: {n_layer}L / {n_head}H / {n_embd}D | block {block_size} | vocab {vocab_size}")

## [C] Load Wikipedia

Download the full `20231101.simple` config to `data/` on first run (parquet via `snapshot_download`, which sidesteps the `datasets` resolution that stalls on this multi-config repo), then load it from local disk — re-runs do no network I/O. With the whole dataset in hand, take a seeded **uniform random sample** of N articles; sampling strategy can be iterated freely since the download + load stay cached. See `01_microgpt-wiki.notes.md`.

In [ ]:
# Anchor a gitignored data/ dir at the repo root (walk up to find pyproject.toml).
_here = Path.cwd().resolve()
repo_root = next(p for p in (_here, *_here.parents) if (p / "pyproject.toml").exists())
data_dir = repo_root / "data"
data_dir.mkdir(exist_ok=True)

# First run: download the full Simple-English config's parquet locally. snapshot_download
# (targeted by allow_patterns) avoids the datasets resolution that stalls on this huge
# multi-config repo. Once the files are on disk, re-runs do no network I/O.
local_dir = data_dir / "wikipedia_simple"
config_dir = local_dir / "20231101.simple"
if not config_dir.exists() or not any(config_dir.glob("*.parquet")):
    print("downloading full 20231101.simple config (first run only) ...")
    snapshot_download(
        repo_id="wikimedia/wikipedia",
        repo_type="dataset",
        allow_patterns="20231101.simple/*.parquet",
        local_dir=str(local_dir),
    )

# Load the local parquet (memory-mapped, random-access) -- fast and offline after step 1.
parquet_files = sorted(str(p) for p in config_dir.glob("*.parquet"))
ds = load_dataset("parquet", data_files=parquet_files, split="train")
print(f"dataset: {len(ds):,} articles from {len(parquet_files)} local parquet file(s)")

# Seeded uniform random sample over the WHOLE dataset; iterate freely by changing
# data_seed / n_articles -- the download + load above stay cached.
idx = random.Random(data_seed).sample(range(len(ds)), n_articles)
corpus = "\n\n".join(ds[i]["text"] for i in idx)
print(f"corpus: {n_articles:,} of {len(ds):,} articles | {len(corpus):,} chars")

## [D] Inspect the corpus

Sanity-check the data: total size, number of articles, and a sample article.

In [ ]:
used = articles[:n_articles]
lengths = [len(a) for a in used]

print(f"articles:        {len(used):,}")
print(f"total chars:     {len(corpus):,}  (~{len(corpus) / 1e6:.1f}M)")
print(f"total bytes:     {len(corpus.encode('utf-8')):,}")
print(f"chars/article:   min {min(lengths):,} | median {sorted(lengths)[len(lengths) // 2]:,} | max {max(lengths):,}")
print(f"distinct chars:  {len(set(corpus)):,}")

# Show one representative article (the median-length one), trimmed.
median_idx = sorted(range(len(used)), key=lambda i: lengths[i])[len(used) // 2]
sample = used[median_idx]
print(f"\n--- sample article (#{median_idx}, {len(sample):,} chars) ---")
print(sample[:800] + ("..." if len(sample) > 800 else ""))

## [E] Train a minimal BPE

Roll our own byte-pair-encoding: the merge loop that learns `vocab_size` merges from the corpus (the minbpe lesson, from scratch).

In [ ]:
# Byte-pair encoding from scratch: start from the 256 raw byte values and
# repeatedly merge the most frequent adjacent pair into a new token.
def get_stats(ids):
    """Count how often each adjacent token pair occurs."""
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts


def merge(ids, pair, idx):
    """Replace every occurrence of `pair` with a single new token `idx` (one L->R pass)."""
    out, i = [], 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            out.append(idx)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out


num_merges = vocab_size - 256        # the 256 byte values are already "tokens"
ids = list(corpus.encode("utf-8"))   # corpus as a flat stream of byte-tokens

merges = {}  # (int, int) -> int, insertion order == merge rank
t0 = time.time()
for k in range(num_merges):
    stats = get_stats(ids)
    pair = max(stats, key=stats.get)  # most frequent adjacent pair
    idx = 256 + k
    ids = merge(ids, pair, idx)
    merges[pair] = idx
print(f"learned {len(merges)} merges in {time.time() - t0:.1f}s")
print(f"final vocab size: {256 + len(merges)} | stream compressed to {len(ids):,} tokens")

## [F] Encode / decode

Implement `encode` (apply merges) and `decode` (invert) and verify a roundtrip on sample text.

In [ ]:
# Build the id -> bytes lookup once, then encode/decode against it.
vocab = {i: bytes([i]) for i in range(256)}
for (p0, p1), idx in merges.items():       # merges is insertion-ordered, so children
    vocab[idx] = vocab[p0] + vocab[p1]     # are always built after their parents


def decode(ids):
    """ids -> text: concatenate each token's bytes, then UTF-8 decode."""
    data = b"".join(vocab[i] for i in ids)
    return data.decode("utf-8", errors="replace")


def encode(text):
    """text -> ids: greedily apply merges in the order they were learned."""
    ids = list(text.encode("utf-8"))
    while len(ids) >= 2:
        stats = get_stats(ids)
        # pick the pair whose merge was learned earliest (lowest rank)
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break                          # nothing left that we know how to merge
        ids = merge(ids, pair, merges[pair])
    return ids


sample = "The quick brown fox jumps over the lazy dog. \U0001f98a"
ids = encode(sample)
roundtrip = decode(ids)
print(f"text:      {sample!r}")
print(f"lengths:   {len(sample.encode('utf-8'))} bytes -> {len(ids)} tokens")
print(f"roundtrip: {roundtrip == sample}\n")

# Show the decoded text with token boundaries marked by underscores.
pieces = [vocab[i].decode("utf-8", errors="replace") for i in ids]
print("_".join(pieces) + "\n")

# Group every multi-character token in the learned vocabulary by length, longest first.
by_len = defaultdict(set)
for idx in vocab:
    p = vocab[idx].decode("utf-8", errors="replace")
    if len(p) > 1:
        by_len[len(p)].add(p)
for length in sorted(by_len, reverse=True):
    toks = sorted(by_len[length])
    print(f"len {length} ({len(toks)}): " + "  ".join(repr(t) for t in toks))

## [G] Tokenize the corpus

Encode the full corpus to token ids and split into train / val tensors.

## [H] Batching

`get_batch`: sample random contiguous (x, y) chunks of length `block_size` for next-token prediction.

## [I] Embeddings

Token + position embedding tables built from raw tensors.

## [J] RMSNorm

Root-mean-square normalization for training stability, from raw tensor ops.

## [K] Multi-head causal self-attention

The heart of the Transformer: Q/K/V projections, scaled dot-product attention with a causal mask, and output projection — all raw tensor math.

## [L] MLP block

Per-position feed-forward block (up-projection, nonlinearity, down-projection).

## [M] Transformer block

Compose one block: RMSNorm + attention + residual, then RMSNorm + MLP + residual.

## [N] GPT model

Assemble embeddings + a stack of blocks + final norm + LM head into a forward pass returning logits and cross-entropy loss.

## [O] Optimizer & LR schedule

Adam optimizer with a decaying learning-rate schedule.

## [P] Training loop

Run training steps with periodic validation-loss evaluation.

## [Q] Loss curves

Plot train / val loss over steps to see learning progress.

## [R] Sampling

Autoregressive generation: sample the next token from the model's distribution with a temperature knob.

## [S] Generate

Produce Wikipedia-style text samples from the trained model.

## [T] Save / load

Persist the BPE merges and model `state_dict` to `artifacts/` (gitignored) so a trained run can be reloaded without retraining.